# Empirical expansion review

## tl;dr
The September 2026 snapshot expands lobbying history to 2003-2008 and acquires 229 reports for four candidate PACs. A reviewed registration alias restores fifteen AdvaMed filings. Separate filing-grain metadata and two original-form reviews reveal incompatible accounting methods and potential income/expense double counting. Of 48 expected committee-half-years, 36 have usable outcomes, including one source-reviewed supplemental-filing decision; twelve remain unresolved. These are source and design inputs, not identified substitution effects. The archived procurement audit finds 243,700 blank award types and 4,505,798 missing offer counts among 6,449,101 rows. A competition-after-exclusion flag is not SAM vendor status. Comment attribution and procurement source/linkage gaps remain unresolved.

## Context & Methods
This offline companion reruns the repository's evidence audit and checks the prepared PAC series against its source report periods. It also distinguishes two public-letter positions from a verified docket copy or observed agency uptake. It makes no network requests and requires no credentials. Run from this notebook's directory or the repository root using Python 3.

### Key Assumptions
Exact-name LDA links, one registration-specific reviewed alias and historical PAC affiliations do not independently validate the actor spine. Native reporting periods are not independent quarters. Repeated issue rows are not separate monetary observations. A missing amount is not zero; API-reported zeros still require original-form adjudication. Organizational expenses may include outside-firm income reports, and reporting methods may cover different activity scopes. Event labels use September 14, 2007 as an enactment convention, not a universal treatment date: House travel restrictions began earlier, so 2007H1 is not an untreated period for that contrast. Read `docs/substitution-study-redesign.md`, `docs/comment-uptake-pilot.md`, and `docs/procurement-source-reconciliation.md` for source URLs and remaining requirements.

## Data
Inputs are the public CSV and JSON snapshots under `data/calibration/first-wave/`. The audit profiles LDA/FEC coverage and adjudications, the comment pilot, and both the GAO discovery worklist and separate four-award linkage pilot. LDA filing metadata, alias reviews and original-form reviews preserve separate source dollars, accounting methods, registration IDs and review scope. The GAO pilot preserves public search responses, projected award details and four archived bulk rows, with source URLs and hashes. Its saved extract is checked against the committed bulk manifest; this notebook does not require or independently re-scan the ignored 6.4-million-row bulk file. Input paths remain repository-relative.

In [1]:
import csv
import importlib.util
import json
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'scripts/audit-empirical-expansion.py').is_file(), 'Run inside this repository'

def load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, root / 'scripts' / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

audit = load_script('expansion_audit', 'audit-empirical-expansion.py')
periods = load_script('fec_periods', 'prepare-substitution-fec-periods.py')
findings = audit.audit()

## Results
### Reproduce the prepared series
Source-marked latest, non-amended reports enter complete half-years, with one explicitly recorded supplemental-loan-paperwork adjudication. The preparation retains every expected half-year and quarantines missing/nonfinite outcomes, gaps, overlaps, straddling reports and unresolved alternatives. The reform-straddling half-year is retained but excluded from event contrasts. The strict source-flag-only sensitivity omits the reviewed exception.

In [2]:
raw_reports = audit.read('substitution-fec-report-panel.csv')
cohort = audit.read('substitution-fec-acquisition-cohort.csv')
adjudications = audit.read('substitution-fec-version-adjudications.csv')
prepared, coverage = periods.prepare_with_coverage(raw_reports, cohort, adjudications)
strict_prepared, _ = periods.prepare_with_coverage(raw_reports, cohort)
assert prepared == audit.read('substitution-fec-halfyear-panel.csv')
assert coverage == audit.read('substitution-fec-halfyear-coverage.csv')
print(f'{len(raw_reports)} raw reports; {len(prepared)} of {len(coverage)} complete half-years; {len(strict_prepared)} without manual adjudication')
for finding in findings:
    print(f"{finding['item']}: {finding['status']}")
    print(finding['evidence'])

229 raw reports; 36 of 48 complete half-years; 35 without manual adjudication
expanded-lda-history: source_only
issueRows=1119; filingUUIDs=427; actors=6; years=2003,2004,2005,2006,2007,2008; observedPrePeriodsByActor=cand-372cc95f9387:9;cand-5d1da86118e5:9;cand-9948f2974958:9;cand-b905d6833296:9;cand-d5522e62fad7:9;cand-f7178708cc78:9
lda-posting-date-anomaly: review_required
postingBeforeCoveredPeriod=3; filingUUIDs=9bd361f7-98e7-46a9-90cd-267ace5ca84c;a93a23e5-9da2-4c18-8625-b41fc0987d06;bcb55688-3d56-4c98-a546-0730eb923bfa
lda-measurement-comparability: source_measures_not_comparable_totals
filings=427; amountKinds={'income': 267, 'neither': 71, 'expenses': 89}; expenseMethodMissing=63; sourceZeroFilings=25; incomeExpenseOverlapActorPeriods=74; aliasFilings=15; reviewedForms=2; reviewedExpenseMethods={'C': 1, 'A': 1}
alternate-channel-reports: bounded_affiliation_candidate
reportRows=229; committees=4; affiliationRows=12; affiliationCycles=2004,2006,2008; overlappingPeriods=56; gap

### Inspect source and reviewed accounting methods separately
One metadata row represents one filing, not one issue or one organization-period. The audit checks exact joins, amount conversion, projected-source fingerprints and alias scope. Original-form method observations remain separate from raw API fields and are not carried across periods. The two reviewed covers are not a representative sample or an amendment-family audit.

In [3]:
metadata = audit.read('substitution-lda-filing-metadata.csv')
filing_reviews = audit.read('substitution-lda-filing-reviews.csv')
metadata_by_uuid = {row['filingUuid']: row for row in metadata}
assert len(metadata_by_uuid) == len(metadata)
for review in filing_reviews:
    source = metadata_by_uuid[review['filingUuid']]
    assert review['sourceFingerprint'] == source['sourceFingerprint']
    method = source['expensesMethod'] or 'missing'
    print(f"{source['primaryName']}: {source['filingYear']} {source['filingPeriod']}; API method {method}; original form method {review['formExpensesMethod']}; reviewed page {review['pagesReviewed']}")

ADVANCED MEDICAL TECHNOLOGY ASSOCIATION: 2003 mid_year; API method missing; original form method C; reviewed page 1
AMERICAN ASSOCIATION FOR JUSTICE: 2003 mid_year; API method missing; original form method A; reviewed page 1


### Reconcile the archived procurement frame
The committed compact profile comes from a full scan of exactly 56 manifest-selected ZIPs, excluding superseded downloads and subaward members. This cell validates saved provenance, category partitions and totals, not the original ZIP bytes. To repeat the source scan, run `python3 scripts/audit-procurement-bulk-frame.py --scan` with the ignored archives available. A separate reviewed raw excerpt links one blank-type EPA record to a current official IDV identity. It does not classify all blank rows or establish contemporaneous 2024 type history.

In [4]:
bulk_audit = load_script('bulk_audit', 'audit-procurement-bulk-frame.py')
manifest = json.loads(bulk_audit.MANIFEST.read_text())
profile = json.loads(bulk_audit.PROFILE.read_text())
totals = bulk_audit.validate_profile(profile, manifest, bulk_audit.sha256(bulk_audit.MANIFEST))
review = json.loads(bulk_audit.TYPE_REVIEW.read_text())
print('Reviewed blank-type example:', bulk_audit.validate_type_review(review, profile))
assert sum(totals['awardTypeRows'].values()) == totals['rows']
assert sum(totals['obligationSignRows'].values()) == totals['rows']
for key in ('rows', 'childDescriptionRows', 'blankAwardTypeRows', 'missingOffersRows', 'zeroOffersRows', 'afterExclusionCompetitionRows'):
    print(f"{key}: {totals[key]:,} ({100 * totals[key] / totals['rows']:.2f}%)")
print('Signed net obligations, dollars:', totals['netObligationDollars'])
print('Absolute obligations, dollars:', totals['absoluteObligationDollars'])
for partition in profile['strata']:
    source = partition['manifestStratum']
    if int(source['rowCountDrift']):
        print(source['agency'], source['startDate'], source['endDate'], 'count/export difference', source['rowCountDrift'])

Reviewed blank-type example: IDV_B_B
rows: 6,449,101 (100.00%)
childDescriptionRows: 6,205,401 (96.22%)
blankAwardTypeRows: 243,700 (3.78%)
missingOffersRows: 4,505,798 (69.87%)
zeroOffersRows: 351 (0.01%)
afterExclusionCompetitionRows: 665,137 (10.31%)
Signed net obligations, dollars: 704337439585.26
Absolute obligations, dollars: 774172847532.38
Department of Agriculture 2024-07-01 2024-09-30 count/export difference -21
Department of Defense 2024-05-01 2024-05-31 count/export difference -1


## Takeaways
The source snapshot contains 1,119 LDA issue rows across 427 filings and six actors, each with nine observed pre-enactment reporting periods. Three filings have posting-date anomalies. Fifteen restored AdvaMed filings include a 2008Q2 amendment that still needs version-family review. The metadata contains 267 income reports, 89 expense reports and 71 filings with neither amount; 63 expense reports lack API accounting methods. Seventy-four actor-period cells contain both reporting sides, which must not be added into organizational totals. The two reviewed 2003 covers use Methods C and A respectively, so being in the same LDA source does not establish a common federal-only outcome.

The four-PAC acquisition frame retains 48 expected half-years: GASPAC, AdvaMed and AAJ each supply twelve, while all twelve Benefits Council periods remain unresolved. AAJ's final half-year depends on the documented supplemental-filing review; omitting that decision yields 35 usable half-years. There are 27 pre-event, three excluded event-straddling and six post-event observations pooled across three PACs. Available outcomes and unassigned reform exposure cannot identify a treatment-control substitution effect. The legacy candidate-contributions field includes Form 3X contributions to other political committees, not solely candidates. The two comment pilot observations have no adjudicated individual-comment links. The 2026 GAO worklist remains unlinked. Separately, four FY2024 award links from one consolidated decision match archived bulk and current original-action records, but none appears in the 28,103-row small panel. That panel contains 134 repeated partial keys, not proven duplicate actions. The pilot supports source reconciliation, not protest rates or a capture effect.

The bulk profile preserves 6,205,401 explicit child-action descriptions and 243,700 unresolved blank types, including one verified vehicle identity. Missing offers cover 69.87% of exported rows and are distinct from 351 explicit source zeros. The 665,137 after-exclusion competition rows describe a competition procedure, not vendor debarment. Agriculture Q4 and Defense May account for the 22-row count/export discrepancy, whose cause and record identities remain unknown. Matching archived hashes and agency/date coverage does not certify a complete, unique-action SAM comparison frame.

This notebook verifies reproducible profiling and transformation, not historical coverage, causal identification, independent manual coding, or representative procurement sampling. It does not alter the Java model or promote calibration claims.